## 1.&nbsp;Import libraries and dataset 💾



In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, cohen_kappa_score, classification_report, roc_auc_score
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.pipeline import make_pipeline
from sklearn.inspection import permutation_importance

from sklearn.compose import make_column_transformer
from sklearn import set_config
set_config(transform_output="pandas")
pd.set_option('future.no_silent_downcasting', True)

file_name = 'data.csv'
if not os.path.exists(file_name):
    file_id = '1CyTOOEYD2pNalZIsq-vU2n6soWf5rlJJ'
    path = f'https://drive.google.com/uc?export=download&id={file_id}'
    df = pd.read_excel(path)
    df.to_csv(file_name, index=False)
df = pd.read_csv(file_name)

df_clean = df.copy()

## 2.&nbsp;Data Cleaning & Initial Preparation

**Initial Data Inspection:**
 - Understand the data structure.
 - Identify missing values.
 - Detect duplicate records.

In [ ]:
df_clean.info()
df_clean.isnull().sum()
df_clean.duplicated().sum()
df_clean['satisfaction_v2'].value_counts()
#df_clean[['Arrival Delay in Minutes', 'Departure Delay in Minutes']].describe()

**Feature Renaming and Target Encoding:**
- Standardizing column names: All features were renamed using the snake_case convention to ensure code consistency, prevent modeling errors, and simplify attribute access.
- Target transformation: The target variable was simplified by merging 'neutral' and 'dissatisfied' categories into a single 'dissatisfied' class.
- Binary encoding: The target was converted into a binary format (1 for 'satisfied', 0 for 'dissatisfied') and explicitly cast to an integer type for machine learning compatibility.


In [ ]:
df_clean = df_clean.rename(columns={'satisfaction_v2':'satisfaction', 'Gender':'gender', 'Customer Type':'customer_type',
                        'Age':'age', 'Type of Travel':'travel_type', 'Class':'class', 'Flight Distance':'distance',
                        'Seat comfort':'seat_comfort', 'Departure/Arrival time convenient':'dep_val_time_convenient',
                        'Food and drink':'food_drink', 'Gate location':'gate', 'Inflight wifi service':'wifi_service',
                        'Inflight entertainment':'entertainment','Online support':'online_support',
                        'Ease of Online booking':'online_booking_service','On-board service':'onboard_service',
                        'Leg room service':'leg_room_service','Baggage handling':'baggage_handling',
                        'Checkin service':'checkin_service','Cleanliness':'cleanliness','Online boarding':'online_boarding',
                        'Departure Delay in Minutes':'departure_delay_minutes','Arrival Delay in Minutes':'arrival_delay_minutes'})

df_clean['customer_type'] = df_clean['customer_type'].replace('disloyal Customer', 'Disloyal Customer')
df_clean['satisfaction'] = df_clean['satisfaction'].replace('neutral or dissatisfied', 'dissatisfied')
df_clean['satisfaction'] = df_clean['satisfaction'].replace({'satisfied': 1, 'dissatisfied': 0})
df_clean['satisfaction'] = df_clean['satisfaction'].astype(int)
df_clean['arrival_delay_minutes'] = df_clean['arrival_delay_minutes'].astype('Int64')

**Initial Data Cleaning and Integrity:**
- Removal of identifiers (id): These columns do not carry any predictive power for the model.
- Removal of duplicates: Executed to prevent model overfitting and ensure data uniqueness.
- Handling missing values: Rows with NaN values in the delay columns were removed, as they account for less than 1% of the total dataset.
- Elimination of redundant features: Departure Delay was excluded due to high multicollinearity with Arrival Delay.

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df_clean,
                x='departure_delay_minutes',
                y='arrival_delay_minutes',
                hue=df_clean['satisfaction'].map({0: 'dissatisfied', 1: 'satisfied'}),
                hue_order=['dissatisfied', 'satisfied'],
                palette='pastel',
                alpha=0.8)

plt.title('Departure vs Arrival Delays', fontsize=14)
plt.xlabel('Departure Delay (min)')
plt.ylabel('Arrival Delay (min)')
plt.legend(title='Satisfaction')
sns.despine()
plt.show()

In [ ]:
df_clean = df_clean.drop('id', axis=1, errors='ignore')
df_clean = df_clean.drop_duplicates()
df_clean = df_clean.dropna(subset=['arrival_delay_minutes'])
df_clean = df_clean.drop('departure_delay_minutes', axis=1, errors='ignore')

df_clean.shape

## 2.&nbsp;Exploratory Data Analysis (EDA)

**Automated Data Profiling:**
- Tool integration: Utilizing ydata-profiling to generate a comprehensive diagnostic report of the cleaned dataset.
- Data health check: Monitoring variable distributions, outliers, interactions, high cardinality warnings to ensure high-quality input for modeling.

In [ ]:
!pip install ydata-profiling
from ydata_profiling import ProfileReport
ProfileReport(df_clean)
profile = ProfileReport(df_clean, title="Airline Passenger Satisfaction Report", explorative=True)
profile.to_notebook_iframe()

**Visualizing Key Relationships:**
- Category insights: Compare satisfaction rates across different traveler types and customer segments to find business-critical patterns.
- Correlation analysis: Identifying key satisfaction drivers through a correlation heatmap.
- Flight delay impact: Evaluate the threshold where arrival delays significantly affect loyalty, and note the presence of outliers.

In [ ]:
df_clean.columns
df_clean['satisfaction'].value_counts(normalize=True).round(4) * 100
df_clean.describe().T

- Flight Delays: Median delay is 0 min; 50% of flights are on time.
- Outliers: Significant spikes in distance and arrival delays (max 1584 min).
- Handling: Preserved all outliers to represent actual passenger experience and prevent model oversimplification.

In [ ]:
# statisfaction & gender distribution (%)
check_cols = ['satisfaction', 'gender']
for col in check_cols:
    print(f"--- {col.upper()} Distribution ---")
    counts = df_clean[col].value_counts()
    percent = df_clean[col].value_counts(normalize=True) * 100

    dist = pd.DataFrame({'Count': counts, 'Percentage (%)': percent})
    print(dist.round(1))
    print("\n")

In [ ]:
# statisfaction & gender distribution (graph)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))

sns.countplot(data=df_clean, x='satisfaction', hue='satisfaction', ax=ax1, palette='pastel', legend=False)
ax1.set_title('Overall Satisfaction Status', fontsize=12, pad=20)
ax1.set(xlabel='', ylabel='Number of Passengers')
ax1.set_xticks([0, 1], ['Dissatisfied', 'Satisfied'])

for c in ax1.containers:
    ax1.bar_label(c, fmt=lambda x: f'{100 * x / len(df_clean):.1f}%', fontweight='bold', padding=3)

sns.countplot(data=df_clean, x='gender', hue='gender', ax=ax2, palette='pastel', legend=False)
ax2.set_title('Gender Distribution', fontsize=12, pad=20)
ax2.set(xlabel='', ylabel='')

for c in ax2.containers:
    ax2.bar_label(c, fmt=lambda x: f'{100 * x / len(df_clean):.1f}%', fontweight='bold', padding=3)

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# gender distribution by key categories (%)
if 'class_combined' not in df_clean.columns:
    df_clean['class_combined'] = df_clean['class'].replace(['Eco', 'Eco Plus'], 'Eco & Eco Plus')
features = ['customer_type', 'class_combined', 'travel_type']
summary_table = []

for feature in features:
    counts = df_clean.groupby([feature, 'gender']).size().unstack()
    percentages = (counts / len(df_clean) * 100).round(1)

    for col in counts.columns:
        counts[f'{col} %'] = percentages[col].astype(str) + '%'

    counts = counts[['Female', 'Female %', 'Male', 'Male %']]
    summary_table.append(counts)

final_df = pd.concat(summary_table)
print("Gender distribution by key categories (%):")
display(final_df)

Dataset Insights:
* Data Balance: Dataset is well-balanced (54.7% satisfied vs. 45.3% dissatisfied); no synthetic balancing (SMOTE) required.
* Gender Balance: The gender distribution is balanced (50.7% female vs. 49.3% male).


In [ ]:
# calculation of the distribution of journey types, classes and customer types by gender (%)
print("--- Split Travel Type by gender (%) ---")
print(pd.crosstab(df_clean['gender'], df_clean['travel_type'], normalize='index').round(4) * 100)
print("\n")

print("--- Split Class by gender (%) ---")
print(pd.crosstab(df_clean['gender'], df_clean['class'], normalize='index').round(4) * 100)
print("\n")

print("--- Split Customer by gender (%) ---")
print(pd.crosstab(df_clean['gender'], df_clean['customer_type'], normalize='index').round(4) * 100)


In [ ]:
# passenger & travel & class type distribution (%)
df_clean['class_combined'] = df_clean['class'].replace(['Eco', 'Eco Plus'], 'Eco & Eco Plus')
check_cols = ['customer_type', 'travel_type', 'class_combined']
for col in check_cols:
    print(f"--- {col.upper()} Distribution ---")
    counts = df_clean[col].value_counts()
    percent = df_clean[col].value_counts(normalize=True) * 100
    dist = pd.DataFrame({'Count': counts, 'Percentage (%)': percent})
    print(dist.round(1))
    print("\n")

In [ ]:
# class vs satisfaction for loyal passenger
loyal_customers = df_clean[df_clean['customer_type'] == 'Loyal Customer']
pd.crosstab(loyal_customers['class'], loyal_customers['satisfaction'], normalize='index') * 100

In [ ]:
# age & passenger & travel & class type distribution (graph)
df_clean['class_combined'] = df_clean['class'].replace(['Eco', 'Eco Plus'], 'Eco & Eco Plus')
cat_cols = ['customer_type', 'class_combined', 'travel_type']
orders = {
    'customer_type': ['Loyal Customer', 'Disloyal Customer'],
    'class_combined': ['Business', 'Eco & Eco Plus'],
    'travel_type': ['Business travel', 'Personal Travel']
}
total = len(df_clean)
fig, axes = plt.subplots(2, 2, figsize=(14, 6))
axes = axes.flatten()

sns.histplot(data=df_clean, x='age', hue='satisfaction', kde=True, ax=axes[0],
             palette='tab10', element="step", legend=False)
axes[0].set_title('Age', fontsize=10)
axes[0].set(xlabel='', ylabel='Passengers')
axes[0].set_xticks(range(0, 85, 10))
axes[0].tick_params(labelsize=8)

for i, col in enumerate(cat_cols):
    ax = axes[i+1]
    order = orders.get(col, df_clean[col].unique())

    sns.countplot(data=df_clean, x=col, hue='satisfaction', ax=ax, order=order, palette='pastel')

    title_name = col.replace("_combined", "").replace("_", " ").title()
    ax.set_title(title_name, fontsize=10)
    ax.set(xlabel='', ylabel='Passengers')
    ax.tick_params(labelsize=8)

    ax.legend(title='Satisfaction', labels=['Dissatisfied', 'Satisfied'],
              fontsize=8, title_fontsize=8, loc='upper right')

    for c in ax.containers:
        ax.bar_label(c, fmt=lambda x: f'{100 * x / total:.1f}%',
                     fontsize=7, fontweight='bold', padding=2)
    ax.margins(y=0.15)

sns.despine()
plt.tight_layout(pad=2.0)
plt.show()

Audience Segments & Loyalty Analysis
* Travel Type: Business travelers are more satisfied than personal travelers, though dissatisfaction rates remain high across both segments.
* Class Advantage: A clear satisfaction gap exists between Business and Eco classes.
* Core Profile: Data confirms the "Ideal Passenger" is a middle-aged professional traveling for business.
* Growth Opportunity: Personal Travel and Economy Class are the primary sources of negative feedback.

In [ ]:
# correlation heatmap: service factors vs satisfaction
corr = df_clean.select_dtypes('number').corr().sort_values('satisfaction', ascending=False)
corr = corr[corr.index]

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5,
            annot_kws={'size': 8}, cbar_kws={'shrink': .8})

plt.title('Correlation Heatmap: Service Factors vs Satisfaction', fontsize=14, pad=20)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(fontsize=10)

plt.tight_layout()
plt.show()

Correlation Matrix Analysis
* Primary Drivers: `entertainment` (0.52) is the undisputed leader of satisfaction, followed by a powerful digital block: `online_booking_service` (0.43) and `online_support` (0.39).
* Digital Synergy: A strong "cluster" is observed between `online_boarding`, `online_booking_service`, and `online_support` (correlations > 0.6). This indicates that passengers perceive the airline's digital ecosystem as a single, unified experience.
* Negative Correlations: `arrival_delay_minutes` and `distance` show negative values. While the impact is relatively low (-0.08 and -0.04), it confirms that longer delays and distances naturally decrease satisfaction levels.
* Weak Predictors: Factors like `age` (0.11), `gate` (-0.01), and `dep_val_time_convenient` (-0.01) show near-zero linear correlation. These features are kept for the ML model to capture potential non-linear interactions.



In [ ]:
# distribution for Entertainment & online booking service (%)
cols_to_analyze = ['entertainment', 'online_booking_service']
for col in cols_to_analyze:
    print(f"\n--- Distribution for: {col.replace('_', ' ').title()} ---")
    tab = pd.crosstab(df_clean[col], df_clean['satisfaction'])
    tab_pct = (tab / len(df_clean) * 100)

    result = pd.concat([tab, tab_pct], axis=1)
    result.columns = ['Count: Dissat', 'Count: Sat', '% Share: Dissat', '% Share: Sat']

    display(result.style.format({
        '% Share: Dissat': '{:.1f}%',
        '% Share: Sat': '{:.1f}%'
    }).background_gradient(cmap='YlGnBu', axis=None))

In [ ]:
# service deep dive: inflight entertainment & seat comfort & online booking (graph)
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(14, 4))
total = len(df_clean)

plots = [
    (ax1, 'entertainment', 'Inflight Entertainment', 'Rating'),
    (ax2, 'seat_comfort', 'Seat Comfort', 'Rating'), # Новый график теперь второй
    (ax3, 'online_booking_service', 'Online Booking', 'Rating')
]
for ax, col, title, xlabel in plots:
    sns.countplot(data=df_clean, x=col, hue='satisfaction', ax=ax, palette='pastel')
    ax.set(title=title, xlabel=xlabel, ylabel='Number of Passengers')
    ax.legend(title='', labels=['Dissatisfied', 'Satisfied'])
    for c in ax.containers:
        ax.bar_label(c, fmt=lambda x: f'{100 * x / total:.1f}%',
                      fontsize=9, fontweight='bold', padding=3)
    ax.margins(y=0.1)

sns.despine()
plt.tight_layout()
plt.show()

Service Deep Dive: Entertainment & Online Booking
* The 4-Star Threshold: For all tree services, satisfaction increases only when the rating hits 4 or 5.
* Entertainment Impact: Satisfaction jumps 6x (from 3.7% to 23.2%) when moving from rating 3 to 4, with top ratings (4-5) holding a 45.1% combined share.
* Seat Comfort: There is also a jump from dissatisfaction to satisfaction at a rating of 4–5.
* Booking Efficiency: High dissatisfaction (11%) persists even at a rating of 3. Loyalty begins at levels 4 and 5 (42.2% combined share).
* Conclusion: Passengers perceive 3-star service almost as negatively as 1-star. To drive satisfaction, the airline must deliver 4-5 stars experiences in these key categories.

In [ ]:
# on-time flights (%)
on_time_pct = (df_clean['arrival_delay_minutes'] == 0).mean() * 100
print(f"Percentage of On-Time Flights: {on_time_pct:.1f}%")

In [ ]:
# delay
len(df_clean[df_clean['arrival_delay_minutes'] > 90])

In [ ]:
len(df_clean[df_clean['arrival_delay_minutes'] > 1500])

In [ ]:
# satisfaction share by delay category & detection of outliers in arrival delays
bins = [-float('inf'), 0, 15, 60, 120, 200, float('inf')]
labels = ['On Time', 'Small (1-15m)', 'Medium (15-60m)', 'Large (60-120m)', 'Long (120-200m)', 'Critical (>200m)']

df_clean['delay_cat'] = pd.cut(df_clean['arrival_delay_minutes'], bins=bins, labels=labels)
ct = pd.crosstab(df_clean['delay_cat'], df_clean['satisfaction'], normalize='index') * 100
ct.columns = ['Dissatisfied', 'Satisfied']
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(22, 7))

ct.plot(kind='bar', stacked=True, ax=ax1, color=sns.color_palette('pastel'), width=0.8, rot=0)
ax1.set_title('Satisfaction Share by Delay Category', fontsize=15, pad=15)
ax1.set_ylabel('Percentage (%)', fontsize=12)
ax1.set_xlabel('')
ax1.set_xticklabels(ct.index, fontweight='bold', fontsize=9)

ax1.axvline(x=3.5, color='red', ls='--', lw=3)
ax1.legend(title='Satisfaction', loc='upper left')

for c in ax1.containers:
    ax1.bar_label(c, fmt='%.1f%%', label_type='center', color='white', fontsize=10)

sns.boxplot(x=df_clean['arrival_delay_minutes'], color='orange', ax=ax2)
ax2.set_title('Detection of Outliers in Arrival Delays', fontsize=15, pad=15)
ax2.set_xlabel('Delay in Minutes', fontsize=12)
ax2.grid(axis='x', linestyle='--', alpha=0.4)

sns.despine()
plt.tight_layout()
plt.show()
df_clean.drop(columns='delay_cat', inplace=True)

Flight Punctuality

* On Time: 56.2%
* Delayed: 43.8%

Delay Impact Analysis
* The "Tipping Point": Passenger satisfaction flips at the 15-minute mark. Once a delay exceeds 15 minutes, the probability of dissatisfaction becomes higher than satisfaction (>52%).
* Resilience Threshold: Up to 60 minutes, the impact is gradual. However, for delays exceeding 120 minutes, dissatisfaction spikes to 63%, a nearly 22% increase compared to on-time flights.
* Diminishing Returns of Delay: The jump from 120 minutes to 200+ minutes shows almost no change in satisfaction (63.1% vs 63.2%).
* Conclusion: For the airline, the most critical "window of save" is the first 60 minutes. After 2 hours, the damage to the brand is already maximized regardless of the further duration.
* Outliers: Extreme delays. Outliers preserved to capture real-world "worst-case" scenarios and ensure model reliability.


# 3. Preprocessing

**Split data**

In [ ]:
# create a dictionary for score comparisons
scores = {}

# Separate target feature from predictor features
X = df_clean.drop('satisfaction', axis=1)
y = df_clean['satisfaction']

# Perform train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

**Preprocessor**

In [ ]:
# build preprocessor
X_num = X_train.select_dtypes(include='number').columns.tolist()
X_ord = ['class']
X_cat = X_train.select_dtypes(exclude='number').columns.difference(X_ord).tolist()
quality_levels = ['Eco', 'Eco Plus', 'Business']
categories_order = [quality_levels for _ in X_ord]

num_pipe = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler())
ord_pipe = make_pipeline(
    SimpleImputer(strategy='most_frequent'),
    OrdinalEncoder(categories=categories_order))
cat_pipe = make_pipeline(
    SimpleImputer(strategy='constant', fill_value='N_A'),
    OneHotEncoder(handle_unknown='ignore', sparse_output=False, drop='if_binary'))

preprocessor = make_column_transformer(
    (num_pipe, X_num),
    (ord_pipe, X_ord),
    (cat_pipe, X_cat)
)
preprocessor

**Modeling**

| **Model** | **Role** | **Rationale for this Dataset** |
| :--- | :--- | :--- |
| **Logistic Regression** | **Baseline** | Establishes a performance floor and tests for simple linear relationships in passenger behavior. |
| **Random Forest** | **Robust Competitor** | An ensemble method that handles non-linearities and high-dimensional data without requiring extensive feature scaling. |
| **XGBoost** | **Advanced Solution** | A state-of-the-art gradient boosting framework, optimized for tabular data and complex feature interactions. |



For the modeling phase, I implemented three different approaches.
* Logistic Regression to establish a baseline.
* Random Forest for its robustness.
* XGBoost.

In [ ]:
# list of models for comparison
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(eval_metric='logloss', random_state=42)
}
# Training and metrics collection
trained_pipelines = {}
results = []
plt.style.use('default')
fig, axes = plt.subplots(1, 3, figsize=(16, 4), gridspec_kw={'wspace': 0.08})

for i, (name, model) in enumerate(models.items()):
    classifier = make_pipeline(preprocessor, model)
    classifier.fit(X_train, y_train)
    trained_pipelines[name] = classifier

    y_pred = classifier.predict(X_test)
    y_probs = classifier.predict_proba(X_test)[:, 1]

    res = {
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_probs)
    }
    results.append(res)

    disp = ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred,
        ax=axes[i],
        cmap='YlGn',
        colorbar=False
        )
    axes[i].set_title(f'{name}', fontsize=10, fontweight='bold', pad=12)
    axes[i].grid(False)

plt.suptitle('Model Comparison: Confusion Matrices', fontsize=12, y=1.03)
plt.show()
results_df = pd.DataFrame(results).sort_values(by='F1-Score', ascending=False)
results_baseline = results_df.copy()
results_baseline['Type'] = 'Baseline'

metrics_to_format = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
results_styled = (results_baseline.style
                  .background_gradient(cmap='YlGn', subset=metrics_to_format)
                  .format({m: lambda x: f"{x*100:.1f}" for m in metrics_to_format})
                  )
display(results_styled)
# display(results_baseline.style.background_gradient(cmap='YlGn').format(precision=4))

* Collinearity Resilience: Random Forest provides well-distributed and stable feature importances, preventing highly correlated attributes from artificially neutralizing each other.
* Robustness to Overfitting: Independence of decision trees ensures structural stability on skewed customer profiles, mitigating the risks of algorithmic overfitting.
* Business Interpretability: Delivers more realistic, transparent, and actionable insight weights for targeted corporate retention campaigns.

Random Forest (Strategic Analytics)
* Collinearity Resilience: Provides well-distributed and stable feature importances, preventing highly correlated attributes from artificially neutralizing each other.
* Robustness to Overfitting: Independence of decision trees ensures structural stability on skewed customer profiles, mitigating algorithmic overfitting risks.

XGBoost (Operational Deployment)
* Production Engine: Implemented as the core pipeline inside the Streamlit web application due to high execution speed.
* Peak Accuracy: Sequential error correction achieves the highest performance metrics.

**Streamlit**

In [ ]:
!pkill streamlit

In [ ]:
# training and saving the model to a .pkl file
import joblib
from google.colab import files
best_pipeline = trained_pipelines['Random Forest']
joblib.dump(best_pipeline, 'airline_rf_pipeline.pkl')
files.download('airline_rf_pipeline.pkl')
print("The pipeline has been saved and is being downloaded!")

In [ ]:
%%writefile app.py
import joblib
import pandas as pd
import streamlit as st
import plotly.graph_objects as go

# 1. Page Configuration & Custom Executive Theme
st.set_page_config(page_title="SkyInsight Executive Dashboard", layout="wide", initial_sidebar_state="expanded")
st.markdown("""
    <style>
    .main { background-color: #fcfcfc; }
    .stSlider > label { font-weight: 600; color: #2c3e50; font-size: 14px; }
    .category-box {
        background-color: #ffffff;
        padding: 20px;
        border-radius: 12px;
        box-shadow: 0 4px 12px rgba(0, 0, 0, 0.04);
        margin-bottom: 20px;
        border-left: 6px solid #ccc;
    }
    .comfort-box { border-left-color: #99ff99; }
    .digital-box { border-left-color: #66b3ff; }
    .service-box { border-left-color: #ff9999; }
    .reliability-box { border-left-color: #ffb84d; }
    .result-card {
        padding: 25px;
        border-radius: 12px;
        text-align: center;
        font-weight: bold;
        font-size: 22px;
        color: white;
        margin-top: 15px;
        margin-bottom: 15px;
    }
    .success-card { background: linear-gradient(135deg, #2ecc71, #27ae60); box-shadow: 0 6px 15px rgba(46, 204, 113, 0.3); }
    .error-card { background: linear-gradient(135deg, #e74c3c, #c0392b); box-shadow: 0 6px 15px rgba(231, 76, 60, 0.3); }
    .action-box {
        background-color: #fff5f5;
        border: 1px dashed #e74c3c;
        padding: 20px;
        border-radius: 8px;
        margin-top: 10px;
    }
    .rec-item {
        margin-bottom: 8px;
        font-size: 14px;
        line-height: 1.4;
    }
    </style>
""", unsafe_allow_html=True)

# 2. Pipeline Loading Function
@st.cache_resource
def load_pipeline():
    return joblib.load('airline_rf_pipeline.pkl')
try:
    pipeline = load_pipeline()
except Exception as e:
    st.error(f"Failed to load the model file 'airline_rf_pipeline.pkl'. Error: {e}")
    st.stop()

# --- MAIN CONTENT AREA ---
st.title("✈️ SkyInsight: Predictive Passenger Experience Analytics")
st.write("Simulate service performance metrics below to compute operational retention forecasting in real-time.")

# 3. High-Level Executive KPIs
st.markdown("### 🎯 System Diagnostic Performance")
kpi1, kpi2, kpi3, kpi4 = st.columns(4)
with kpi1: st.metric(label="Overall Accuracy", value="96%")
with kpi2: st.metric(label="Model Reliability (AUC)", value="99%")
with kpi3: st.metric(label="Prediction Precision", value="97%")
with kpi4: st.metric(label="Risk Detection (Recall)", value="96%")
st.markdown("---")

# 3. Passenger Survey User Interface
st.header("📋 Passenger Survey")

# Demographics and Flight Context (Basic Info)
col1, col2 = st.columns(2)
with col1:
    gender = st.selectbox("Passenger Gender", ["Male", "Female"])
    age = st.number_input("Passenger Age", min_value=1, max_value=120, value=35)
with col2:
    travel_type = st.selectbox("Type of Travel", ["Business travel", "Personal Travel"])
    flight_class = st.selectbox("Class", ["Eco", "Eco Plus", "Business"], index=0)

left_layout, right_layout = st.columns(2, gap="large")
with left_layout:
    st.markdown("### 🛠️ Service Touchpoint Simulator")

    # CATEGORY 1: InFlight Comfort
    st.markdown('<div class="category-box comfort-box">', unsafe_allow_html=True)
    st.markdown("#### 🛋️ InFlight Comfort")
    c1, c2 = st.columns(2)
    with c1:
        entertainment = st.slider("Inflight entertainment", 1, 5, 3)
        seat_comfort = st.slider("Seat comfort", 1, 5, 3)
        food_drink = st.slider("Food and drink", 1, 5, 3)
    with c2:
        cleanliness = st.slider("Cleanliness", 1, 5, 3)
        leg_room_service = st.slider("Leg room service", 1, 5, 3)
    st.markdown('</div>', unsafe_allow_html=True)

    # CATEGORY 2: Digital Experience
    st.markdown('<div class="category-box digital-box">', unsafe_allow_html=True)
    st.markdown("#### 💻 Digital Experience")
    d1, d2 = st.columns(2)
    with d1:
        online_booking_service = st.slider("Ease of online booking", 1, 5, 3)
        online_boarding = st.slider("Online boarding", 1, 5, 3)
    with d2:
        wifi_service = st.slider("Inflight wi-fi service", 1, 5, 3)
        online_support = st.slider("Online support", 1, 5, 3)
    st.markdown('</div>', unsafe_allow_html=True)

    # CATEGORY 3: Airport & Crew Service
    st.markdown('<div class="category-box service-box">', unsafe_allow_html=True)
    st.markdown("#### 🧳 Airport & Crew Service")
    a1, a2 = st.columns(2)
    with a1:
        onboard_service = st.slider("Onboard service", 1, 5, 3)
        checkin_service = st.slider("Check-in service", 1, 5, 3)
    with a2:
        baggage_handling = st.slider("Baggage handling", 1, 5, 3)
    st.markdown('</div>', unsafe_allow_html=True)

    # CATEGORY 4: Flight Reliability
    st.markdown('<div class="category-box reliability-box">', unsafe_allow_html=True)
    st.markdown("#### ⏱️ Flight Reliability & Delays")
    r1, r2 = st.columns(2)
    with r1:
        gate = st.slider("Gate location convenience", 1, 5, 3)
        dep_val_time_convenient = st.slider("Time convenience", 1, 5, 3)
    with r2:
        departure_delay_minutes = st.number_input("Departure Delay (min)", min_value=0, value=0)
        arrival_delay_minutes = st.number_input("Arrival Delay (min)", min_value=0, value=0)
    st.markdown('</div>', unsafe_allow_html=True)

with right_layout:
    st.markdown("### 📊 Experience Geometry & Scoring")
    comfort_avg = (entertainment + seat_comfort + food_drink + cleanliness + leg_room_service) / 5
    digital_avg = (online_booking_service + online_boarding + wifi_service + online_support) / 4
    service_avg = (onboard_service + checkin_service + baggage_handling) / 3
    reliability_avg = (gate + dep_val_time_convenient) / 2

    fig = go.Figure()
    fig.add_trace(go.Scatterpolar(
        r=[comfort_avg, digital_avg, service_avg, reliability_avg, comfort_avg],
        theta=['InFlight Comfort', 'Digital Experience', 'Airport & Crew', 'Flight Reliability', 'InFlight Comfort'],
        fill='toself',
        fillcolor='rgba(102, 179, 255, 0.2)',
        line=dict(color='#66b3ff', width=3),
        name='Current Service Profile'
    ))
    fig.update_layout(
        polar=dict(radialaxis=dict(visible=True, range=[1, 5])),
        showlegend=False,
        margin=dict(l=40, r=40, t=20, b=20),
        height=340
    )
    st.plotly_chart(fig, use_container_width=True)

    # 4. Dataframe Construction for Prediction
    input_data = pd.DataFrame([{
        'gender': gender,
        'age': age,
        'travel_type': travel_type,
        'customer_type': "Loyal Customer",
        'class': flight_class,
        'distance': 1800,
        'seat_comfort': seat_comfort,
        'dep_val_time_convenient': dep_val_time_convenient,
        'food_drink': food_drink,
        'gate': gate,
        'wifi_service': wifi_service,
        'entertainment': entertainment,
        'online_support': online_support,
        'online_booking_service': online_booking_service,
        'onboard_service': onboard_service,
        'leg_room_service': leg_room_service,
        'baggage_handling': baggage_handling,
        'checkin_service': checkin_service,
        'cleanliness': cleanliness,
        'online_boarding': online_boarding,
        'departure_delay_minutes': int(departure_delay_minutes),
        'arrival_delay_minutes': float(arrival_delay_minutes)
    }])

    try:
        expected_cols = pipeline.feature_names_in_
        input_data = input_data[expected_cols]
    except:
        pass

    # 5. Prediction & Custom Threshold Execution (Панель перенесена направо)
    st.markdown("### 📈 Risk Evaluation Execution")
    retention_threshold = st.slider(
        "Minimum Satisfaction Threshold (%)",
        min_value=50, max_value=90, value=65, step=5,
        help="The minimum probability required to consider a passenger safely retained."
    )

    if st.button("Execute Strategic Predictive Scoring", type="primary", use_container_width=True):
        probabilities = pipeline.predict_proba(input_data)
        satisfied_prob = probabilities[0][1] * 100

        if satisfied_prob >= retention_threshold:
            st.markdown(f"""
                <div class="result-card success-card">
                    🎉 PASSENGER RETENTION<br>
                    <span style="font-size: 15px; font-weight: normal;">Satisfaction Probability: {satisfied_prob:.1f}% | Status: CHURN PROBABILITY NEUTRAL</span>
                </div>
            """, unsafe_allow_html=True)
            st.balloons()
        else:
            st.markdown(f"""
                <div class="result-card error-card">
                    ⚠️ PASSENGER CHURN RISK<br>
                    <span style="font-size: 15px; font-weight: normal;">Dissatisfaction Probability: {100 - satisfied_prob:.1f}% | Risk: CHURN PROBABILITY POSSIBLE</span>
                </div>
            """, unsafe_allow_html=True)

            st.markdown("### ⚙️ Risks of passenger Dissatisfaction")
            recommendations = []

            if entertainment <= 3:
                recommendations.append("<div class='rec-item'>• 📺 <b>Inflight Entertainment Friction.</b></div>")
            if seat_comfort <= 3:
                recommendations.append("<div class='rec-item'>• 🛋️ <b>Seat Comfort Issue.</b></div>")
            if food_drink <= 3:
                recommendations.append("<div class='rec-item'>• 🍱 <b>Catering Dissatisfaction.</b></div>")
            if cleanliness <= 3:
                recommendations.append("<div class='rec-item'>• 🧼 <b>Cabin Cleanliness Deficit.</b></div>")
            if leg_room_service <= 3:
                recommendations.append("<div class='rec-item'>• 🦿 <b>Leg Room Discomfort.</b></div>")

            if online_booking_service <= 3 or online_boarding <= 3:
                recommendations.append("<div class='rec-item'>• 📱 <b>Digital Interface Friction.</b></div>")
            if wifi_service <= 3:
                recommendations.append("<div class='rec-item'>• 🌐 <b>Inflight Wi-Fi Failure.</b></div>")
            if online_support <= 3:
                recommendations.append("<div class='rec-item'>• ☎️ <b>Substandard Online Support.</b></div>")

            if onboard_service <= 3:
                recommendations.append("<div class='rec-item'>• 🧑‍✈️ <b>Crew Service Failure.</b></div>")
            if checkin_service <= 3:
                recommendations.append("<div class='rec-item'>• 🛎️ <b>Check-in Service Issues.</b></div>")
            if baggage_handling <= 3:
                recommendations.append("<div class='rec-item'>• 🧳 <b>Baggage Handling Friction.</b></div>")

            if gate <= 3:
                recommendations.append("<div class='rec-item'>• 🚪 <b>Inconvenient Gate Location.</b></div>")
            if dep_val_time_convenient <= 3:
                recommendations.append("<div class='rec-item'>• 📅 <b>Schedule Inconvenience.</b></div>")

            if arrival_delay_minutes > 15 or departure_delay_minutes > 15:
                max_delay = max(arrival_delay_minutes, departure_delay_minutes)
                recommendations.append(f"<div class='rec-item'>• ⏱️ <b>Operational Delay ({max_delay} min)</b></div>")

            if not recommendations:
                recommendations.append("<div class='rec-item'>• 🔄 <b>Composite Experience Friction:</b> Issue a standard 1,000 miles goodwill compensation and escalate account to the prioritized customer retention tier.</div>")

            st.markdown(f'<div class="action-box">{"".join(recommendations)}</div>', unsafe_allow_html=True)


In [ ]:
!pip install pyngrok -q
import time
from pyngrok import ngrok
!pip install streamlit pyngrok -q

!pkill streamlit
ngrok.kill()

NGROK_TOKEN = ""
ngrok.set_auth_token(NGROK_TOKEN)
os.system("nohup streamlit run app.py > streamlit.log 2>&1 &")

print("Installing dependencies and starting Streamlit server...")
time.sleep(4)
try:
    public_url = ngrok.connect(8501, proto="http")
    print("="*50)
    print(f" Your application is live at: {public_url.public_url}")
    print("="*50)
except Exception as e:
    print(f"Ngrok connection failed: {e}")

**Feature Importance**

In [ ]:
# feature importance (only 14 features)
service_cols = [
    "seat_comfort",
    "dep_val_time_convenient",
    "food_drink",
    "gate",
    "wifi_service",
    "entertainment",
    "online_support",
    "online_booking_service",
    "onboard_service",
    "leg_room_service",
    "baggage_handling",
    "checkin_service",
    "cleanliness",
    "online_boarding",]
category_map = {
    "entertainment": "InFlight Comfort",
    "seat_comfort": "InFlight Comfort",
    "food_drink": "InFlight Comfort",
    "cleanliness": "InFlight Comfort",
    "leg_room_service": "InFlight Comfort",
    "onboard_service": "Airport & Crew Service",
    "checkin_service": "Airport & Crew Service",
    "baggage_handling": "Airport & Crew Service",
    "online_booking_service": "Digital Experience",
    "online_boarding": "Digital Experience",
    "wifi_service": "Digital Experience",
    "online_support": "Digital Experience",
    "gate": "Flight Reliability",
    "dep_val_time_convenient": "Flight Reliability",}
colors = {
    "InFlight Comfort": "#99ff99",
    "Digital Experience": "#66b3ff",
    "Flight Reliability": "#ffb84d",
    "Airport & Crew Service": "#ff9999",}
def clean_name(name):
    return name.replace("_", " ").title()

X_train_clean = X_train[service_cols].copy()
X_test_clean = X_test[service_cols].copy()
preprocessor = make_column_transformer(
    (OneHotEncoder(drop="first", sparse_output=False), []),
    remainder="passthrough",)
clean_pipeline = make_pipeline(
    preprocessor, RandomForestClassifier(random_state=42))
clean_pipeline.fit(X_train_clean, y_train)

feature_names = clean_pipeline.named_steps[
    "columntransformer"
].get_feature_names_out()
clean_names = [name.split("__")[-1] for name in feature_names]

importances = clean_pipeline.named_steps[
    "randomforestclassifier"
].feature_importances_
all_feature_imp_df = pd.DataFrame(
    {"Feature": clean_names, "Importance": importances})
filtered_feature_imp_df = all_feature_imp_df[
    all_feature_imp_df["Feature"].isin(service_cols)
].copy()
if filtered_feature_imp_df["Importance"].sum() > 0:
    filtered_feature_imp_df["Importance"] = (
        filtered_feature_imp_df["Importance"]
        / filtered_feature_imp_df["Importance"].sum()) * 100
filtered_feature_imp_df["Category"] = filtered_feature_imp_df["Feature"].map(
    category_map)
filtered_feature_imp_df["Feature"] = filtered_feature_imp_df["Feature"].apply(
    clean_name)

filtered_feature_imp_df = filtered_feature_imp_df.sort_values(
    by="Importance", ascending=False
).reset_index(drop=True)
extended_labels = {}
for cat, color in colors.items():
    cat_features = [clean_name(f) for f, c in category_map.items() if c == cat]
    features_str = ", ".join(cat_features)
    if len(features_str) > 40:
        mid = len(cat_features) // 2
        features_str = (
            ", ".join(cat_features[:mid])
            + ",\n"
            + ", ".join(cat_features[mid:]))
    extended_labels[cat] = f"{cat}\n[{features_str}]"

filtered_feature_imp_df["Legend_Label"] = filtered_feature_imp_df[
    "Category"].map(extended_labels)
extended_colors = {
    extended_labels[cat]: color for cat, color in colors.items()}

fig, ax = plt.subplots(figsize=(10, 7))
sns.barplot(
    x="Importance",
    y="Feature",
    data=filtered_feature_imp_df,
    hue="Legend_Label",
    palette=extended_colors,
    dodge=False,
    ax=ax,)
ax.set_yticks(range(len(filtered_feature_imp_df)))
ax.set_yticklabels(filtered_feature_imp_df["Feature"], fontsize=10)
ax.set_xlim(0, filtered_feature_imp_df["Importance"].max() * 1.15)
#for container in ax.containers:
#    ax.bar_label(
#        container, fmt="%.1f%%", fontsize=9, padding=5)
plt.title(
    "Feature Importance",
    fontsize=12,
    pad=15)
plt.xlabel("", fontsize=10)
plt.ylabel("")
legend = plt.legend(
    loc="center right",
    bbox_to_anchor=(1.0, 0.65),
    labelspacing=1.4,
    frameon=True,
    facecolor="white",
    edgecolor="none",)
for text in legend.get_texts():
    lines = text.get_text().split("\n")
    if len(lines) > 1:
        title_with_spaces = lines[0].replace(" ", "\\ ")
        formatted_text = f"$\\bf{{{title_with_spaces}}}$\n{lines[1]}"
        if len(lines) > 2:
            formatted_text += f"\n{lines[2]}"
        text.set_text(formatted_text)
    text.set_fontsize(9)
    text.set_color("#333333")

sns.despine(left=True, bottom=True)
plt.grid(axis="x", linestyle="--", alpha=0.3)
plt.tight_layout()
plt.show()

df_print = filtered_feature_imp_df[["Feature", "Importance"]].copy()
df_print.columns = ["Признак", "Важность (%)"]
print(
    df_print.to_string(
        index=False, formatters={"Важность (%)": "{:,.2f}%".format}))

* Strategic Priority: Retention of loyal passengers depends heavily on InFlight Comfort and Digital Experience.
* Key Drivers: The primary satisfaction drivers are Entertainment, Seat Comfort, and Online Services.
* Hidden Value: Convenient departure times and optimized gate proximity are latent but important factors.

**Hyperparameter Optimization**

In [ ]:
# optimization XGBoost parameter
param_grid = {
    'xgbclassifier__max_depth': [8, 10],
    'xgbclassifier__learning_rate': [0.05, 0.1],
    'xgbclassifier__n_estimators': [500, 1000],
    'xgbclassifier__colsample_bytree': [0.8],
    'xgbclassifier__min_child_weight': [1, 5],
    'xgbclassifier__gamma': [0.1, 0.2]}
base_pipeline = make_pipeline(preprocessor, XGBClassifier(tree_method='hist',
                              device='cpu', eval_metric='logloss', random_state=42))
grid_search = GridSearchCV(
    base_pipeline,
    param_grid,
    cv=3,
    scoring='f1',
    verbose=2,
    n_jobs=-1)
grid_search.fit(X_train, y_train)
print(f"\nBest psrsmeter: {grid_search.best_params_}")

best_model = grid_search.best_estimator_
y_pred_final = best_model.predict(X_test)
y_probs_final = best_model.predict_proba(X_test)[:, 1]

final_report = pd.DataFrame([{
    'Model': 'XGBoost (Tuned)',
    'Accuracy': accuracy_score(y_test, y_pred_final),
    'Precision': precision_score(y_test, y_pred_final),
    'Recall': recall_score(y_test, y_pred_final),
    'F1-Score': f1_score(y_test, y_pred_final),
    'ROC-AUC': roc_auc_score(y_test, y_probs_final)
}])
display(final_report.style.format(precision=4))
print("\nDetailed classification report:")
report = classification_report(y_test, y_pred_final, target_names=['Dissatisfied', 'Satisfied'])
print(report)

**Final Confusion Matrix**

In [ ]:
best_model = grid_search.best_estimator_
fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_final,
    ax=ax,
    display_labels=['Dissatisfied', 'Satisfied'],
    cmap='YlGn',
    values_format='d' )

plt.title(f'Final Confusion Matrix: XGBoost (Tuned)\nF1-Score: {f1_score(y_test, y_pred_final):.2f}',
          fontsize=12, pad=20)
plt.grid(False)
plt.show()

total = len(y_test)
errors = (y_test != y_pred_final).sum()
print(f"Total number of tests: {total}")
print(f"Model errors: {errors} ({errors/total:.2%})")

* False Negatives: 604
* False Positives: 379

**Churn Analysis**

In [ ]:
# the gap in ratings between satisfied and dissatisfied loyal passengers
service_cols = ['seat_comfort', 'dep_val_time_convenient',
       'food_drink', 'gate', 'wifi_service', 'entertainment', 'online_support',
       'online_booking_service', 'onboard_service', 'leg_room_service',
       'baggage_handling', 'checkin_service', 'cleanliness', 'online_boarding']
actual_service_cols = [col for col in service_cols if col in df_clean.columns]

loyal_dissatisfied = df_clean[(df_clean['customer_type'] == 'Loyal Customer') & (df_clean['satisfaction'] == 0)]
loyal_satisfied = df_clean[(df_clean['customer_type'] == 'Loyal Customer') & (df_clean['satisfaction'] == 1)]
comparison = pd.DataFrame({
    'Dissatisfied': loyal_dissatisfied[actual_service_cols].mean(),
    'Satisfied': loyal_satisfied[actual_service_cols].mean()
})
comparison['Diff'] = comparison['Satisfied'] - comparison['Dissatisfied']
print("The gap in ratings between satisfied and dissatisfied loyal passengers.")
print(comparison['Diff'].sort_values(ascending=False).head(15).round(2))

In [ ]:
# the gap in ratings between satisfied and dissatisfied loyal passengers (graph)
comparison['Diff'] = comparison['Satisfied'] - comparison['Dissatisfied']
comparison_sorted = comparison.sort_values(by='Diff', ascending=False)
plt.figure(figsize=(10, 6))
sns.barplot(
    x=comparison_sorted['Diff'],
    y=comparison_sorted.index,
    palette='Reds_r'
)
plt.title('Анализ разрыва в оценках лояльных пассажиров\n(Satisfied vs Dissatisfied)', fontsize=12, fontweight='bold')
plt.xlabel('Величина разрыва (Разница средних оценок)')
plt.ylabel('')
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# the gap in ratings between loyal and disloyal passengers all
loyal_passengers = df_clean[df_clean['customer_type'] == 'Loyal Customer']
disloyal_passengers = df_clean[df_clean['customer_type'] == 'Disloyal Customer']

comparison = pd.DataFrame({
    'Disloyal Customers': disloyal_passengers[actual_service_cols].mean(),
    'Loyal Customers': loyal_passengers[actual_service_cols].mean()})
comparison['Diff'] = comparison['Loyal Customers'] - comparison['Disloyal Customers']

print("The gap in ratings between loyal and disloyal all passengers.")
print(comparison['Diff'].sort_values(ascending=False).head(5).round(2))

* Insight: The services with the greatest difference are the key drivers of satisfaction. If their ratings drop, the customer is bound to become dissatisfied.
The widest gap between satisfied and dissatisfied loyal customers is found in [entertainment]. This means that this particular service is the decisive factor: if we improve it, we will win back the loyalty of 80% of our churning customers.

In [ ]:
# loyal customers are dissatisfied: "danger zone"
y_pred = trained_pipelines['Random Forest'].predict(X_test)
analysis_df = X_test.copy()
analysis_df['actual'] = y_test
analysis_df['predicted'] = y_pred
loyal_upset = analysis_df[
    (analysis_df['customer_type'] == 'Loyal Customer') &
    (analysis_df['actual'] == 0)]

service_cols = ['wifi_service', 'online_booking_service', 'online_boarding', 'seat_comfort',
                'gate', 'entertainment', 'dep_val_time_convenient',  'onboard_service', 'leg_room_service',
                'cleanliness', 'food_drink', 'checkin_service', 'baggage_handling', 'online_support']
low_scores = loyal_upset[service_cols].mean().sort_values()
print(f"Loyal but dissatisfied passengers account for {len(loyal_upset) / len(analysis_df[analysis_df['actual'] == 0]):.1%} of the total number of dissatisfied passengers.")

for service, score in low_scores.items():
    print(f"{service:.<30} {score:.2f}")
avg_delay = loyal_upset['arrival_delay_minutes'].mean()
delayed_passengers_pct = (loyal_upset['arrival_delay_minutes'] > 0).mean()
print(f"\n--- The factor of flight delays ---")
print(f"Average arrival delay: {avg_delay:.1f} мин.")
print(f"The proportion of passengers whose flight was delayed by at least one minute: {delayed_passengers_pct:.1%}")

successfully_detected = loyal_upset[loyal_upset['predicted'] == 0]
recall_loyal_upset = len(successfully_detected) / len(loyal_upset)
print(f"\n--- The effectiveness of the customer retention model ---")
print(f"Accuracy of identifying customers at risk: {recall_loyal_upset:.1%}") # Accuracy of identifying customers at risk (Recall)

In [ ]:
# loyal customers are dissatisfied: "danger zone" (graph)
at_risk_data = {
    'Seat Comfort': 2.5,
    'Entertainment': 2.7,
    'Online Booking': 2.8,
    'Online Boarding': 2.8,
    'Food & Drink': 2.8,
    'Wifi Service': 2.9,
    'Onboard Service': 2.9,
    'Checkin Service': 2.94,
    'Leg Room': 3.0,
    'Gate Location': 3.0,
    'Baggage Handling': 3.3,
    'Cleanliness': 3.3,
    'Departure/Arrival Convenience': 3.4}
df_risk = pd.DataFrame(list(at_risk_data.items()), columns=['Service', 'Score'])
plt.figure(figsize=(10, 4.2))
colors = sns.color_palette("Reds_r", n_colors=len(df_risk))
ax = sns.barplot(x='Score', y='Service', data=df_risk, palette=colors, hue='Service', legend=False)
plt.title('Loyal customers are dissatisfied: "Danger Zone"', fontsize=14, pad=16)

plt.xlabel('Rating', fontsize=10)
plt.ylabel('')
plt.xlim(0, 5)

plt.axvline(x=3.0, color='black', linestyle='--', alpha=0.5)
plt.text(3.1, 0.5, '', color='black', alpha=0.7)
for i, v in enumerate(df_risk['Score']):
    ax.text(v + 0.1, i, str(v), color='black', va='center')
plt.axvspan(0, 3, color='red', alpha=0.05)

sns.despine()
plt.tight_layout()
plt.show()

* High Alert: Features rated below 3 represent a churn risk zone for loyal passengers.
* Critical: The first 3 key features are at risk. The target performance metric for these parameters is 4+ .
* Action Plan: Upgrade cabin seating and In-Flight Entertainment. Accelerate investments in digital & online service infrastructure.

In [ ]:
%%writefile requirements.txt
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, cohen_kappa_score, classification_report, roc_auc_score
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.pipeline import make_pipeline
from sklearn.inspection import permutation_importance

from sklearn.compose import make_column_transformer
from sklearn import set_config
set_config(transform_output="pandas")
pd.set_option('future.no_silent_downcasting', True)